# Session 5: Building MCP Server & Client

**Learning goals:**

- Explain how MCP reduces repeated tool integration work in AI applications.
- Identify the host, client and server, and distinguish tools, resources and prompts.
- Compare stdio with Streamable HTTP and trace a tool discovery and invocation exchange.
- Build a weather MCP server and connect a Python client to it.
- Design a fixed MCP resource, a resource template and a prompt with named arguments.

**Prerequisites:**

- Python functions, dictionaries, JSON, HTTP and Function Calling.
- Use a WeatherAPI key for the custom weather tool.
- Have an LLM API key and a model that supports Function Calling.


## Outline

1. [From Function Calling to MCP](#from-function-calling-to-mcp)
2. [MCP Architecture and Core Primitives](#mcp-architecture-and-core-primitives)
3. [Transport in MCP](#transport-in-mcp)
4. [Build Your Own MCP Server](#build-your-own-mcp-server)
5. [Connect MCP to the Chatbot](#connect-mcp-to-the-chatbot)
6. [Multi-Tool MCP Chatbot](#multi-tool-mcp-chatbot)
7. [Adding Prompt and Resource Features](#adding-prompt-and-resource-features)


## 1. From Function Calling to MCP

In [Sessions 3–4](<../03-04-Building-Chatbot-with-LLM-API/Session 3-4 Building Chatbot with LLM API.ipynb>), we implemented a weather function, wrote its JSON Schema, and added a function mapping and dispatch code. Each application needed its own tool integration code. With more applications and tools, that work was repeated.

The **Model Context Protocol (MCP)** provides a common interface for discovering and invoking external capabilities. Tool implementations live in reusable servers, which compatible applications access through MCP clients.

- **Discover tools:** `tools/list` returns tool names, descriptions and input schemas. FastMCP can generate these schemas from Python type hints and docstrings.
- **Call tools:** `tools/call` sends a tool name and arguments to the server for execution.
- **Reuse implementations:** multiple applications can connect to the same tool server instead of maintaining separate copies of its service-specific code.

**Function Calling and MCP work together:** the model proposes a tool call; the host checks permissions and routes it through MCP. The host still converts tool descriptions to its LLM provider's format and returns results with the matching call ID.

In the figure, both designs use the same two APIs. Without MCP, two applications maintain four adapter implementations. With MCP, they reuse two adapters held in shared servers. The orange blocks show where the service-specific code lives.

[Official MCP architecture](https://modelcontextprotocol.io/docs/learn/architecture)


![Applications duplicate service-specific adapters without MCP; with MCP, they reuse adapters in shared servers.](images/before-after-mcp.png)


## 2. MCP Architecture and Core Primitives

| Role | Responsibility | Connection to earlier sessions |
|---|---|---|
| Host | Owns the user interface, LLM conversation, server selection and permissions | The chatbot application from Sessions 3–4 |
| Client | Handles protocol communication with one server | A protocol-aware connection inside the chatbot |
| Server | Publishes and implements tools, resources and prompts | Similar service responsibility to Session 2's FastAPI backend |

One host can create several clients, each with a one-to-one connection to a server. A server process can also serve other clients when its transport and deployment support that. The LLM does not directly execute Python: the host routes an allowed request through a client.

![An AI host contains two clients, each connected to one server exposing tools, resources and prompts.](images/mcp-architecture.png)

In the diagram, Client A connects to the weather server over stdio. Client B connects to an illustrative document server over Streamable HTTP, labelled HTTP in the figure. Each server groups its capabilities into tools, resources and prompts. A server may expose any supported subset; it does not have to provide all three. The weather examples match the server we build later; the document server illustrates a second connection. Arrows from the LLM represent requests routed by the host, which controls execution.

The host controls which servers and data enter a conversation. For example, a document server and a weather server need separate permissions even when they share the same protocol. A server can wrap a REST API internally; MCP does not replace that upstream API.

The practical weather example below uses HTTP so the server and client can run in separate terminals.


### 2.1 Tools, Resources and Prompts

| Primitive | What it offers | Common control pattern | Example |
|---|---|---|---|
| Tool | An executable operation, including reads or writes | Model proposes; host authorises and invokes | `get_weather(city="Perth")` |
| Resource | Addressable context read by URI | Application selects context | `info://server-status` |
| Prompt | A reusable message template with arguments | User selects a workflow through the host | A weather briefing template |

Resources provide data without a write operation; their content can still change over time. A resource URI identifies content and need not be a web URL. Resources may be fixed (`info://server-status`) or templated (`docs://documents/{doc_id}`). Text, JSON and binary data use appropriate MIME types.

A prompt can produce one or more messages. Retrieving it does not call the LLM or run its suggested tools. The host decides how to use those messages.

Tools are our main focus because they connect to the previous chatbot. We will also read one resource and retrieve one prompt to make the distinctions concrete.

[MCP server concepts](https://modelcontextprotocol.io/docs/learn/server-concepts)


![The MCP client invokes tools, reads resources and retrieves prompts exposed by a server; each capability is illustrated with examples.](images/mcp-client-server-primitives.png)


## 3. Transport in MCP

A **transport** handles how messages are sent and received between an MCP client and server.

1. **For servers running locally:**
   - **stdio (standard input/output):** the client starts the server as a subprocess and exchanges messages through its standard input and output.

2. **For remote servers:**
   - **HTTP+SSE (Server-Sent Events):** the older transport from protocol version **2024-11-05**, using HTTP POST for client messages and a separate SSE connection for server messages.
   - **Streamable HTTP:** introduced in protocol version **2025-03-26**, using one MCP endpoint with JSON or SSE responses. It replaces HTTP+SSE.

HTTP servers can also run locally, as in the two-terminal examples.

[Official MCP transport specification](https://modelcontextprotocol.io/specification/2026-07-28/basic/transports)

![Compare local subprocess pipes and an independent HTTP server.](images/mcp-transport.png)

**Communication lifecycle: two protocol versions**

Both versions use `tools/list` to discover available tools and `tools/call` to invoke one. The main difference is how protocol information is exchanged before and during those requests.

- **MCP 2025-11-25:** the client first sends `initialize`, receives the server's version and capabilities, then sends `notifications/initialized`. It can then list and call tools. Communication ends by closing the transport.
- **MCP 2026-07-28:** there is no required initialisation handshake. The client can optionally call `server/discover`; each request carries its protocol version and client capabilities. HTTP requests complete independently, with no protocol session to terminate. A local stdio connection ends by closing its pipes.

Clients supporting both versions can use the older handshake when connecting to a legacy server. The lower row is a teaching sequence, rather than three mandatory protocol phases.

[Session-based flow (2025-11-25)](https://modelcontextprotocol.io/specification/2025-11-25/basic/lifecycle) · [Per-request metadata and compatibility (2026-07-28)](https://modelcontextprotocol.io/specification/2026-07-28/basic/versioning)

![MCP 2025-11-25 uses an initialisation handshake; MCP 2026-07-28 uses per-request metadata. Both versions list and call tools.](images/mcp-communication-lifecycle.png)


## 4. Build Your Own MCP Server

Install the required libraries in your Python environment:

```bat
python -m pip install fastmcp requests
```

Use the existing WeatherAPI request in `get_weather(city)` to return `city`, `current_temp` and `condition`. The `@mcp.tool()` decorator exposes the function through MCP and generates its input schema.

The function's **docstring** (`"""..."""`) becomes part of the MCP tool definition:

- The summary describes what the tool does, helping the LLM decide when to call it.
- The `Args:` section describes each input, helping the LLM supply suitable values such as `city`.

FastMCP reads these descriptions automatically. Type hints such as `city: str` define the parameter types. The function name supplies the tool identifier; the default display title comes from that name, not the docstring. Changing the docstring changes the published descriptions, while the function body determines the returned weather data.

The server also includes a status resource and a weather briefing prompt.


In [ ]:
"""Expose WeatherAPI through a local MCP server for Session 5."""

import requests
from fastmcp import FastMCP

WEATHER_API_KEY = "YOUR_WEATHERAPI_KEY"  # Replace with your WeatherAPI key.
CURRENT_WEATHER_URL = "https://api.weatherapi.com/v1/current.json"
mcp = FastMCP(name="Course Weather Server")


@mcp.tool()
def get_weather(city: str) -> dict:
    """Return simplified weather for a city using WeatherAPI.

    Args:
        city: A city name, such as Perth or Sydney; include the country if needed.
    """
    city = city.strip()
    if not city:
        raise ValueError("Enter a city name.")
    api_key = WEATHER_API_KEY.strip()
    if not api_key or api_key == "YOUR_WEATHERAPI_KEY":
        raise ValueError("Replace WEATHER_API_KEY in weather_mcp_server.py before calling get_weather.")

    # key authenticates the request; q selects a location; aqi omits air-quality data.
    params = {"key": api_key, "q": city, "aqi": "no"}
    response = requests.get(
        CURRENT_WEATHER_URL, params=params, timeout=10  # Wait up to 10 seconds.
    )
    response.raise_for_status()
    data = response.json()

    # Flatten the location and current objects into one small record.
    return {
        "city": data["location"]["name"],
        "current_temp": data["current"]["temp_c"],
        "condition": data["current"]["condition"]["text"],
    }


@mcp.resource("info://server-status", mime_type="text/plain")
def server_status() -> str:
    """Describe this server without making a weather request."""
    return "Course Weather Server. Data source: WeatherAPI. A key is needed for live weather."


@mcp.prompt()
def weather_briefing(city: str) -> str:
    """Prepare a reusable request for a short weather briefing."""
    return f"Use current weather for {city} to write a short briefing. Include the city, temperature and conditions."


**Run the server in terminal**

The notebook cell defines the server; use either command below to start it.

On the TAFE Windows computers, run [setup_mcp_ev.bat](practice/setup_mcp_ev.bat) to reuse the `api_ss` folder and its `.venv` environment, creating them if needed. [Download weather_mcp_server.py](practice/weather_mcp_server.py) into your working directory. Set `WEATHER_API_KEY` in the file and activate the environment. Inspector requires Node.js 22.19.0 or newer, which includes `npx`.

**1. Inspector — stdio transport**

Run this command in your working directory. Inspector starts the server and connects through stdio:

```bat
npx @modelcontextprotocol/inspector python weather_mcp_server.py -- --transport stdio
```

Open the browser URL printed by Inspector, connect to the server, and call `get_weather` with `city` set to `Perth`.

**2. Inspector — http transport**

Start the server in Terminal 1:

```bat
python weather_mcp_server.py
```

Keep it running. The server uses Streamable HTTP at `http://127.0.0.1:8000/mcp`. In Terminal 2, run Inspector:

```bat
npx @modelcontextprotocol/inspector --server-url http://127.0.0.1:8000/mcp --transport http
```

Open the browser URL printed by Inspector, connect to the server, and test `get_weather`. In section 5, use the same HTTP server with the Python client or chatbot. Press **Ctrl+C** in each terminal to stop its process.

[FastMCP HTTP server guide](https://gofastmcp.com/deployment/running-server) · [MCP Inspector guide](https://modelcontextprotocol.io/docs/2026-07-28/tools/inspector)


## 5. Connect MCP to the Chatbot

The chatbot sends messages to an LLM API and calls `get_weather` through MCP. The FastMCP server requests WeatherAPI data and returns it to the chatbot for the LLM's answer. The diagram shows the HTTP connection.

![The chatbot connects to an LLM API and a FastMCP weather server, which calls the Weather API.](images/mcp-weather-chatbot-connection.png)

### 5.1 MCP Client

Set `MCP_TRANSPORT` to `"http"` or `"stdio"`. HTTP requires a running weather server; stdio starts the server automatically.


In [ ]:
import json
import sys
from fastmcp import Client
from fastmcp.client.transports import StdioTransport

SERVER_URL = "http://127.0.0.1:8000/mcp"
MCP_TRANSPORT = "http"  # Choose "http" or "stdio" for the notebook examples.
SERVER_SCRIPT = "practice/weather_mcp_server.py"
server_target = SERVER_URL if MCP_TRANSPORT == "http" else StdioTransport(
    command=sys.executable,
    args=[SERVER_SCRIPT, "--transport", "stdio"],
    keep_alive=False,  # Close the server subprocess after each client session.
)

def tool_schema(tool):
    """Read the input schema exposed by the installed MCP SDK."""
    if hasattr(tool, "input_schema"):
        return tool.input_schema
    return tool.inputSchema

async with Client(server_target) as client:
    weather_tools = await client.list_tools()  # Discover this server's tools.
    for tool in weather_tools:
        print(tool.name, json.dumps(tool_schema(tool), indent=2))


The client discovers `get_weather` and its input schema. Set the WeatherAPI key in the server file, then call the tool below. Restart an HTTP server after changing its file.


In [ ]:
def result_text(result):
    """Join the text content returned by the weather MCP tool."""
    return "\n".join(block.text for block in result.content if getattr(block, "type", None) == "text")

async with Client(server_target) as client:
    result = await client.call_tool("get_weather", {"city": "Perth"})  # Request current weather.
    print(result_text(result))


### 5.2 Integrate into Chatbot

`MCPChatbot` keeps the `conversation`, streaming response handling and tool-result messages from [Sessions 3–4](<../03-04-Building-Chatbot-with-LLM-API/Session 3-4 Building Chatbot with LLM API.ipynb>). The tool definitions now come from MCP, and `execute_tool()` calls the MCP server in place of the local function mapping.

Set `LLM_PROVIDER` to `gemini` or `openrouter` and enter the matching API key. Gemini uses the same model as the Sessions 3–4 weather chatbot script. The OpenRouter example uses its Free Models Router, which selects a model that supports the request. You can set `CHAT_MODEL` to a specific model that supports tool calling. [Gemini model guide](https://ai.google.dev/gemini-api/docs/models/gemini-3.6-flash) · [OpenRouter Free Models Router](https://openrouter.ai/openrouter/free)


In [ ]:
import asyncio
import requests

LLM_PROVIDER = "gemini"  # Choose "gemini" or "openrouter".

if LLM_PROVIDER == "gemini":
    CHAT_URL = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
    CHAT_API_KEY = "YOUR_GEMINI_API_KEY"
    CHAT_MODEL = "gemini-3.6-flash"
elif LLM_PROVIDER == "openrouter":
    CHAT_URL = "https://openrouter.ai/api/v1/chat/completions"
    CHAT_API_KEY = "YOUR_OPENROUTER_API_KEY"
    CHAT_MODEL = "openrouter/free"  # Selects a free model that supports the request.
else:
    raise ValueError('Choose "gemini" or "openrouter" for LLM_PROVIDER.')


Read streamed text and tool-call fragments from the LLM response.


In [ ]:
def read_stream_deltas(response):
    """Read text and tool fragments from the API's SSE data lines."""
    # A small chunk_size avoids buffering short replies before displaying them.
    for line in response.iter_lines(chunk_size=1, decode_unicode=True):
        if not line.startswith("data:"):
            continue  # Skip blank lines and keep-alive comments.
        data = line[5:].strip()
        if data == "[DONE]":
            return
        chunk = json.loads(data)
        if chunk.get("error"):
            raise RuntimeError(chunk["error"])
        for choice in chunk.get("choices", []):
            if choice.get("finish_reason") not in (None, "stop", "tool_calls"):
                raise RuntimeError(f"Incomplete model response: {choice['finish_reason']}")
            yield choice.get("delta", {})
    raise RuntimeError("The model response stream ended before [DONE].")


**Connecting to MCP and using its tools**

- `connect_and_run()` checks the settings, enters `async with self.connect_to_server()` and runs `chat_loop()` within that context.
- `connect_to_server()` opens the FastMCP client, discovers the tools once and stores their definitions in `self.available_tools`. The connection stays open for the chat and closes when the chat loop exits.
- `list_tools_for_llm()` calls `self.client.list_tools()` to send a `tools/list` request. It converts each tool's name, description and input schema into the LLM API's function format. `send_messages()` includes those definitions in the `tools` field.
- `execute_tool(tool_name, tool_args)` checks that the requested name is in `tool_names`. It calls `self.client.call_tool()`, which sends the name and arguments in a `tools/call` request. The method extracts the text from the MCP result and returns it to `chat()`.

**Follow a weather question through the client**

The chatbot is the host. It uses its FastMCP client to exchange MCP messages with the weather server over stdio or Streamable HTTP. It talks to the LLM API separately, using HTTP requests with streamed responses. Read the diagram from top to bottom.

![Sequence diagram showing the chatbot discovering tools from the MCP server, sending tool definitions to the LLM, calling the requested MCP tool and returning its result to the LLM for a final answer.](images/mcp-client-message-flow.png)

1. You ask **What is the weather in Perth?** The chatbot sends the conversation and available tool definitions to the LLM API. The LLM can respond with a `tool_calls` entry requesting `get_weather` with `{"city": "Perth"}`.
2. `chat()` reads the tool name and decodes the JSON arguments, then passes them to `execute_tool()`. The FastMCP client sends the call to the weather MCP server. The server runs `get_weather` and sends the result back to the client.
3. `chat()` adds a message with `role: tool` and the matching `tool_call_id`. It sends the updated conversation to the LLM, which can use the weather result to stream an answer. If the LLM asks for another tool call, `chat()` repeats the same exchange.

**Using resources and prompts over the same connection**

The weather server also exposes a resource and a prompt. While `connect_to_server()` keeps the client open, the chatbot can fetch either through the same client. `list_tools_for_llm()` prepares the tool definitions for the LLM API. The chatbot decides when to add retrieved resource content or rendered prompt messages to the conversation.

- **Resources** use `list_resources()` to discover resources and `read_resource("info://server-status")` to read this server's status. [FastMCP resource guide](https://gofastmcp.com/docs/clients/resources)
- **Prompts** use `list_prompts()` to discover templates and `get_prompt("weather_briefing", {"city": "Perth"})` to render this server's weather briefing prompt. [FastMCP prompt guide](https://gofastmcp.com/docs/clients/prompts)


In [ ]:
from contextlib import asynccontextmanager


class MCPChatbot:
    """Own the MCP connection, LLM settings and conversation history."""

    def __init__(self, transport="http", *, server_url=SERVER_URL,
                 server_script="weather_mcp_server.py", api_url=CHAT_URL,
                 api_key=CHAT_API_KEY, model=CHAT_MODEL,
                 system_prompt=None):
        self.transport = transport
        self.api_url, self.api_key, self.model = api_url, api_key, model
        server = server_url if transport == "http" else StdioTransport(
            command=sys.executable,  # Use this Python environment for the subprocess.
            args=[server_script, "--transport", "stdio"],
            keep_alive=False,  # Close the subprocess when the client disconnects.
        )
        self.client = Client(server)
        self.available_tools = []
        self.conversation = [{"role": "system", "content": system_prompt or (
            "You are a weather assistant. Use Australian English. "
            "Do not invent weather data. Ask for a city when it is missing."
        )}]

    def validate_settings(self):
        """Check the transport, API endpoint, API key and model before connecting."""
        if self.transport not in ("http", "stdio"):
            raise ValueError("Choose http or stdio for the MCP transport.")
        for name, value in (
            ("CHAT_URL", self.api_url),
            ("CHAT_API_KEY", self.api_key),
            ("CHAT_MODEL", self.model),
        ):
            if not isinstance(value, str) or not value.strip() or value.strip().startswith("YOUR_"):
                raise ValueError(f"Replace {name} before starting the chatbot.")
        if not self.api_url.startswith(("https://", "http://")):
            raise ValueError("CHAT_URL must be an HTTP or HTTPS API endpoint.")

    async def list_tools_for_llm(self):
        """List MCP tools and convert them to the LLM's function format."""
        tools = await self.client.list_tools()
        return [
            {"type": "function", "function": {
                "name": tool.name,
                "description": tool.description or tool.name,
                "parameters": tool_schema(tool),
            }}
            for tool in tools
        ]

    async def execute_tool(self, tool_name, tool_args):
        """Check the discovered tool name, invoke MCP and return text to the LLM."""
        tool_names = {tool["function"]["name"] for tool in self.available_tools}
        if tool_name not in tool_names:
            raise ValueError(f"Tool was not discovered on this server: {tool_name}")
        result = await self.client.call_tool(tool_name, tool_args)
        return result_text(result)

    @asynccontextmanager
    async def connect_to_server(self):
        """Open the configured MCP client and discover its tools."""
        async with self.client:
            self.available_tools = await self.list_tools_for_llm()
            tool_names = {tool["function"]["name"] for tool in self.available_tools}
            if not tool_names:
                raise ValueError("The server did not publish any tools.")
            print("Connected to server with tools:", sorted(tool_names))
            yield  # Keep the client open until the caller leaves this context.

    async def connect_and_run(self):
        """Validate settings and run the chat while the MCP client is open."""
        self.validate_settings()
        async with self.connect_to_server():
            await self.chat_loop()

    def send_messages(self, messages):
        """Print streamed text and return the assembled assistant message."""
        # stream requests incremental responses from the model API.
        payload = {"model": self.model, "messages": messages, "tools": self.available_tools, "stream": True}
        message = {"role": "assistant", "content": ""}
        tool_calls = {}
        with requests.post(
            self.api_url,
            headers={"Authorization": f"Bearer {self.api_key}"},
            json=payload,
            stream=True,  # Read the HTTP response incrementally.
            timeout=(10, 60),  # Connection timeout and wait between received data.
        ) as response:
            response.raise_for_status()
            response.encoding = "utf-8"
            for delta in read_stream_deltas(response):
                text = delta.get("content")
                if text:
                    if not message["content"]:
                        print("Assistant: ", end="", flush=True)
                    print(text, end="", flush=True)
                    message["content"] += text

                # Group fragments by index; parallel tool calls may be interleaved.
                for fragment in delta.get("tool_calls") or []:
                    index = fragment.get("index", 0)
                    if index not in tool_calls:
                        tool_calls[index] = {
                            "id": "",
                            "type": "function",
                            "function": {"name": "", "arguments": ""},
                        }
                    call = tool_calls[index]
                    for key, value in fragment.items():
                        if key not in ("index", "function"):
                            # Preserve provider metadata, including thought signatures.
                            call[key] = value
                    function = fragment.get("function") or {}
                    for key in ("name", "arguments"):
                        call["function"][key] += function.get(key) or ""
        if message["content"]:
            print(flush=True)
        if tool_calls:
            message["tool_calls"] = [tool_calls[index] for index in sorted(tool_calls)]
        return message

    async def chat(self, user_message: str) -> str:
        """Complete a chat turn, running requested tools through MCP."""
        pending = self.conversation + [{"role": "user", "content": user_message}]
        while True:
            # Keep the MCP event loop responsive while requests reads the LLM stream.
            response = await asyncio.to_thread(self.send_messages, pending)
            pending.append(response)  # Preserve the text, tool calls and their IDs.
            if not response.get("tool_calls"):
                break  # A text answer with no tool calls completes the turn.
            for call in response["tool_calls"]:
                tool_name = call["function"]["name"]
                tool_args = json.loads(call["function"]["arguments"])  # Read the JSON arguments.
                print(f"Calling tool {tool_name} with args {tool_args}", flush=True)
                result = await self.execute_tool(tool_name, tool_args)  # Invoke the MCP tool.
                pending.append({
                    "role": "tool",
                    "tool_call_id": call["id"],
                    "content": result,  # Return the MCP text to the LLM.
                })

        self.conversation = pending  # Save the complete turn after the requests succeed.
        return response["content"]

    async def chat_loop(self):
        """Read terminal input and complete turns until the user types quit."""
        print("\nMCP Chatbot Started!\n")
        print("Type your queries or 'quit' to exit.")
        while True:
            user_message = (await asyncio.to_thread(input, "You: ")).strip()
            if user_message.lower() == "quit":
                print("Chat ended.")
                return
            if user_message:
                await self.chat(user_message)


Create the chatbot. On startup, it prints the available tools and the prompt to enter a query or `quit`.


In [ ]:
chatbot = MCPChatbot(
    transport=MCP_TRANSPORT,  # Reuse the transport selected in section 5.1.
    server_script=SERVER_SCRIPT,
)


### 5.3 Run in Terminal

Save [mcp_chatbot.py](practice/mcp_chatbot.py) and [weather_mcp_server.py](practice/weather_mcp_server.py) in your working directory. Enter the LLM settings and WeatherAPI key in their respective files. Activate the Python environment.

**HTTP — two terminals**

Terminal 1:

```bat
python weather_mcp_server.py --transport http
```

Terminal 2 connects to `http://127.0.0.1:8000/mcp`:

```bat
python mcp_chatbot.py --transport http
```

**stdio — one terminal**

The chatbot manages the server subprocess:

```bat
python mcp_chatbot.py --transport stdio
```

Ask about Perth, then Sydney. Type `quit` to exit; stop an HTTP server with **Ctrl+C**.

For notebook chat, select `MCP_TRANSPORT`, enter the LLM settings and enable the cell below.

For direct tool calls, use [mcp_client.py](practice/mcp_client.py) (HTTP) or [mcp_client_stdio.py](practice/mcp_client_stdio.py) (stdio).


In [ ]:
RUN_CHAT = False  # Set True to enable interactive LLM and weather API requests.

if RUN_CHAT:
    await chatbot.connect_and_run()  # Use await in a notebook that already has an event loop.
else:
    print("Interactive chat is off. Set RUN_CHAT to True when you are ready.")


## 6. Multi-Tool MCP Chatbot

This chatbot combines weather enquiries and research-paper search in one conversation. The `MultiToolChatbot` class inherits the streaming chat and tool handling from `MCPChatbot`, using four tools over HTTP or stdio.

| Tool | Functionality |
|---|---|
| `get_weather(city)` | Current temperature and conditions |
| `get_forecast(city, days=1)` | Daily minimum and maximum temperatures for one to three days |
| `get_history(city, dt)` | Daily recorded temperatures for a date in `YYYY-MM-DD` format |
| `search_papers(query, max_results=5)` | arXiv titles, summaries and source links; one to ten results |

**Download and configure**

- [mcp_chatbot.py](practice/mcp_chatbot.py): provides the shared `MCPChatbot` class. Select `LLM_PROVIDER` and enter its API key in this file.
- [multi_tool_mcp_server.py](practice/multi_tool_mcp_server.py): enter your WeatherAPI key in `WEATHER_API_KEY`.
- [mcp_multi_tool_chatbot.py](practice/mcp_multi_tool_chatbot.py): runs the chatbot using the LLM settings from `mcp_chatbot.py`.

Save all three files in the same working directory. The existing `setup_mcp_ev.bat` installs `fastmcp`, `requests` and `arxiv`. No arXiv API key is needed. Forecast and historical access depend on your WeatherAPI plan; choose a supported past date for history requests.

**stdio — one terminal**

Activate the Python environment and run:

```bat
python mcp_multi_tool_chatbot.py --transport stdio
```

The chatbot starts and closes the multi-tool server subprocess.

**HTTP — two terminals**

Start the multi-tool server in Terminal 1:

```bat
python multi_tool_mcp_server.py --transport http
```

In Terminal 2, connect the chatbot to `http://127.0.0.1:8001/mcp`:

```bat
python mcp_multi_tool_chatbot.py --transport http
```

Type `quit` to end the chat. For HTTP, press **Ctrl+C** in Terminal 1 to stop the server.

**Try these requests**

1. “What is the current weather in Perth?” — `get_weather`.
2. “Give me a three-day forecast for Sydney.” — `get_forecast`.
3. “What was the weather in Perth yesterday?” — `get_history`.
4. “Find two arXiv papers about weather forecasting.” — `search_papers`.
5. “Compare current weather in Perth and Sydney.” — two `get_weather` calls.

Watch the printed tool names and arguments. Follow up with another city, date or topic to see how the chatbot keeps the conversation history and chooses a different tool.


## 7. Adding Prompt and Resource Features

Extend the weather server with reference data and reusable instructions. Reuse the three WeatherAPI tools from Section 6, then add resources and prompts.


### 7.1 Comparing Tools, Resources and Prompts

| Dimension | Tools (`@mcp.tool`) | Resources (`@mcp.resource`) | Prompts (`@mcp.prompt`) |
|---|---|---|---|
| **Primary purpose** | Enable the model to perform dynamic actions, computations and queries | Provide passive, **read-only** reference data and domain context | Provide reusable, structured prompt workflows and templates |
| **Initiator / Controller** | **Model-driven**: The LLM autonomously decides when and how to invoke tools | **Host- or User-driven**: The application or user selects context to attach | **User- or Host-driven**: The user triggers pre-configured workflow templates |
| **Addressing scheme** | Named function identifiers (e.g. `get_weather`) | Standardised URI or URI template (e.g. `weather://reference/cities`) | Named template identifier (e.g. `travel_weather_planner`) |
| **Protocol methods** | `tools/list`, `tools/call` | `resources/list`, `resources/read`, `resources/templates/list` | `prompts/list`, `prompts/get` |


#### Why Not Make Everything a Tool?

1. **Direct reads.** The client can fetch a resource by URI without waiting for a model to choose a tool.
2. **Read-only access.** Resources give the client a separate way to retrieve reference data.
3. **Reusable instructions.** Prompts let users apply the same instructions with different inputs.


### 7.2 Weather Tools

Reuse `get_weather`, `get_forecast` and `get_history` from the [Section 6 server](practice/multi_tool_mcp_server.py). Register them with `@mcp.tool()` as before. These tools call WeatherAPI. Set `WEATHER_API_KEY` to try them. 

In [ ]:
from datetime import date

import requests
from fastmcp import FastMCP

WEATHER_API_KEY = "YOUR_WEATHERAPI_KEY"  # Replace with your WeatherAPI key.
CURRENT_WEATHER_URL = "https://api.weatherapi.com/v1/current.json"
mcp = FastMCP(name="Weather Server")


@mcp.tool()
def get_weather(city: str) -> dict:
    """Get current observed weather in degrees Celsius, not a forecast or history.

    Args:
        city: City name, such as Perth, Australia.
    """
    # key authenticates the request; q selects a location; aqi omits air-quality data.
    params = {"key": WEATHER_API_KEY, "q": city.strip(), "aqi": "no"}
    response = requests.get(CURRENT_WEATHER_URL, params=params, timeout=10)
    response.raise_for_status()
    data = response.json()
    return {
        "city": data["location"]["name"],
        "current_temp": data["current"]["temp_c"],
        "condition": data["current"]["condition"]["text"],
    }


FORECAST_WEATHER_URL = "https://api.weatherapi.com/v1/forecast.json"


def shape_daily_weather(data: dict) -> dict:
    """Keep the daily fields shared by forecast and history responses."""
    return {
        "city": data["location"]["name"],
        "days": [
            {"date": day["date"], "max": day["day"]["maxtemp_c"], "min": day["day"]["mintemp_c"]}
            for day in data["forecast"]["forecastday"]
        ],
    }


@mcp.tool()
def get_forecast(city: str, days: int = 1) -> dict:
    """Get a daily forecast for today and following days in Celsius, not history.

    Args:
        city: City name, such as Sydney, Australia.
        days: Number of forecast days, from one to three.
    """
    if not 1 <= days <= 3:
        raise ValueError("Choose one to three forecast days.")
    # days sets the forecast horizon; q selects a location.
    params = {"key": WEATHER_API_KEY, "q": city.strip(), "days": days}
    response = requests.get(FORECAST_WEATHER_URL, params=params, timeout=10)
    response.raise_for_status()
    return shape_daily_weather(response.json())

HISTORY_WEATHER_URL = "https://api.weatherapi.com/v1/history.json"


@mcp.tool()
def get_history(city: str, dt: str) -> dict:
    """Get recorded weather for a past date, subject to the WeatherAPI plan.

    Args:
        city: City name, such as Perth, Australia.
        dt: A past calendar date in YYYY-MM-DD format, not a forecast date.
    """
    history_date = date.fromisoformat(dt)  # Parse the YYYY-MM-DD input.
    params = {"key": WEATHER_API_KEY, "q": city.strip(), "dt": history_date.isoformat()}
    response = requests.get(HISTORY_WEATHER_URL, params=params, timeout=10)
    response.raise_for_status()
    return shape_daily_weather(response.json())


### 7.3 Weather Resources

Use `@mcp.resource()` to publish data from local files. Open the notebook from the Session 5 folder so it can find the `sample_data/` directory beside it.

| Resource URI | File | What the server returns |
|---|---|---|
| `weather://reference/cities` | [reference_cities.json](sample_data/reference_cities.json){download="reference_cities.json"} | All city records as a Markdown table |
| `weather://reference/cities/{city}` | `reference_cities.json` | One city record formatted as Markdown |
| `weather://reference/guidelines/{category}` | [activity.md](sample_data/activity.md){download="activity.md"} or [comfort.md](sample_data/comfort.md){download="comfort.md"} | The selected guide's Markdown text |

Both city resources read the same JSON file. For a guide, the `{category}` value selects a known filename. Each function opens its file when the client reads the resource.


In [ ]:
import json

SAMPLE_DATA_DIR = "sample_data"
GUIDELINE_FILES = {
    "activity": "activity.md",
    "comfort": "comfort.md",
}


def load_reference_cities() -> dict:
    """Load the city records shared by both city resources."""
    with open(f"{SAMPLE_DATA_DIR}/reference_cities.json", encoding="utf-8") as source:
        return json.load(source)


@mcp.resource("weather://reference/cities", mime_type="text/markdown")
def reference_cities() -> str:
    """Return a catalogue of Australian reference cities."""
    header = """# Australian Reference Cities

| City | State | Climate | Coordinates |
|---|---|---|---|
"""
    cities = load_reference_cities()
    rows = [
        f"| {city['name']} | {city['state']} | {city['climate']} | {city['coordinates']} |"
        for city in cities.values()
    ]
    return header + "\n".join(rows)


@mcp.resource("weather://reference/cities/{city}", mime_type="text/markdown")
def reference_city(city: str) -> str:
    """Return the reference entry for one city."""
    cities = load_reference_cities()
    key = city.strip().lower()
    if key not in cities:
        raise ValueError("Choose a city from weather://reference/cities.")
    record = cities[key]
    return (
        f"# {record['name']}, {record['state']}\n\n"
        f"- Climate: {record['climate']}\n"
        f"- Coordinates: {record['coordinates']}"
    )


@mcp.resource("weather://reference/guidelines/{category}", mime_type="text/markdown")
def reference_guidelines(category: str) -> str:
    """Return a Markdown guide for the requested category."""
    key = category.strip().lower()
    if key not in GUIDELINE_FILES:
        raise ValueError("Choose activity or comfort.")
    # Map the category to a known file in the data directory.
    filename = GUIDELINE_FILES[key]
    with open(f"{SAMPLE_DATA_DIR}/{filename}", encoding="utf-8") as source:
        return source.read()


**Read a resource from the client**

`Client(mcp)` connects directly to the server object you created in this notebook. The `async with` block opens the connection and closes it when the example finishes.

First, list the fixed resource URIs with `list_resources()` and the URI patterns with `list_resource_templates()`. Then replace `{city}` with `perth` and pass the resulting URI to `read_resource()`. The server loads `reference_cities.json`, looks up `perth` and returns the city entry as Markdown. The second request reads the text of `comfort.md` through the guidelines URI.


In [ ]:
from fastmcp import Client

async with Client(mcp) as client:
    # Discover the fixed URI separately from the templates.
    print("Fixed resources:", [str(item.uri) for item in await client.list_resources()])
    print("URI templates:", [item.uri_template for item in await client.list_resource_templates()])

    # Fill the city template with a concrete value to read the entry.
    city_entry, = await client.read_resource("weather://reference/cities/perth")
    print(city_entry.text)

    # Use the same read operation to retrieve a Markdown file.
    comfort_guide, = await client.read_resource("weather://reference/guidelines/comfort")
    print(comfort_guide.text)


### 7.4 Weather Prompts

Use `@mcp.prompt()` to register a Python function that returns instructions. Its parameters become the prompt arguments. `weather_briefing()` requires a city and defaults to `general travellers` for its audience. The other templates cover travel planning and historical comparisons.


In [ ]:
@mcp.prompt()
def weather_briefing(city: str, audience: str = "general travellers") -> str:
    """Prepare an audience-tailored weather briefing using current observations and guidelines."""
    return (
        f"You are a weather specialist preparing a concise briefing for {audience} in {city}.\n"
        f"1. Call get_weather(city={city!r}) to inspect current observed temperatures and conditions.\n"
        "2. Consult weather://reference/guidelines/comfort for appropriate temperature thresholds.\n"
        "3. Deliver a concise 2-3 sentence briefing with practical attire and activity recommendations."
    )


@mcp.prompt()
def travel_weather_planner(destination: str, forecast_days: int = 3, activity_type: str = "sightseeing") -> str:
    """Prepare a multi-day itinerary plan based on forecast predictions and activity guidelines."""
    return (
        f"Plan an outdoor activity schedule for {activity_type} in {destination} over {forecast_days} days.\n"
        f"1. Call get_forecast(city={destination!r}, days={forecast_days}) to inspect daily forecasts.\n"
        "2. Consult weather://reference/guidelines/activity to evaluate outdoor weather suitability.\n"
        "3. Provide a structured day-by-day plan with optimal morning/afternoon timing."
    )


@mcp.prompt()
def weather_anomaly_check(city: str, past_date: str) -> str:
    """Evaluate whether current conditions deviate unusually from a historical date."""
    return (
        f"Conduct a weather comparison for {city} between today and {past_date}.\n"
        f"1. Call get_history(city={city!r}, dt={past_date!r}) to obtain historical baseline temperatures.\n"
        f"2. Call get_weather(city={city!r}) to obtain current temperatures today.\n"
        "3. Compare the two records, noting temperature differences and notable shifts in conditions."
    )


**Render a prompt from the client**

Connect to the same server with `Client(mcp)`. Use `list_prompts()` to see the registered template names, then call `get_prompt()` with `weather_briefing` and a dictionary containing the city and audience.

The server fills in the template and returns messages with roles and content. The client prints the `user` message for a briefing aimed at photography students in Perth. This step only retrieves the instructions, so you can inspect them before sending them to a model.


In [ ]:
from fastmcp import Client

async with Client(mcp) as client:
    print("Available prompts:", [prompt.name for prompt in await client.list_prompts()])

    # Render the instructions with a city and a specific audience.
    briefing = await client.get_prompt(
        "weather_briefing",
        arguments={"city": "Perth", "audience": "photography students"},
    )
    for message in briefing.messages:
        print(f"[{message.role}] {message.content.text}")


### 7.5 Client Commands

Extend the `MCPChatbot` from Section 5 with a new `chat_loop()`. The terminal prints the available shortcuts at startup and prompts with `Query:`.

- `@cities` reads the catalogue. `@perth` reads the Perth entry, and `@guidelines comfort` reads the comfort guide.
- `/prompts` lists the prompt names and arguments. `/prompt <name> <arg=value>` renders a prompt and passes it to the existing chat method. Quote values such as `"photography students"` to keep the words together. Enter `/prompt` by itself to see its usage.
- `/tools` and `/call` list and invoke tools directly. Ordinary questions go through the existing chat and tool-calling flow.

Start with the command handler. It uses `query.startswith("@")` and `query.startswith("/")` to choose the MCP request and returns text for the terminal to display.


In [ ]:
import shlex
from urllib.parse import quote

from fastmcp import Client
from mcp.types import TextContent, TextResourceContents


async def handle_command(client: Client, query: str) -> str:
    """Translate a terminal command into an MCP resource, prompt, or tool request."""
    # Translate @aliases into resource URIs before reading them.
    if query.startswith("@"):
        target = query[1:].strip().lower()
        if not target:
            raise ValueError("Use @cities, @perth or @guidelines <activity|comfort>.")
        parts = target.split(maxsplit=1)
        if target == "cities":
            uri = "weather://reference/cities"
        elif parts[0] == "guidelines" and len(parts) == 2:
            category = quote(parts[1], safe="")
            uri = f"weather://reference/guidelines/{category}"
        elif len(parts) == 1 and target != "guidelines":
            city = quote(target, safe="")
            uri = f"weather://reference/cities/{city}"
        else:
            raise ValueError("Use @cities, @perth or @guidelines <activity|comfort>.")

        contents = await client.read_resource(uri)
        return "\n".join(block.text for block in contents if isinstance(block, TextResourceContents))

    elif query.startswith("/"):
        # Keep quoted multi-word argument values together.
        command, *parts = shlex.split(query)
        command = command.lower()
        if command == "/prompts" and not parts:
            prompts = await client.list_prompts()
            lines = []
            for prompt in prompts:
                arguments = [
                    f"{arg.name}=VALUE" if arg.required else f"[{arg.name}=VALUE]"
                    for arg in (prompt.arguments or [])
                ]
                lines.append(f"{prompt.name} " + " ".join(arguments))
            return "\n".join(lines) or "The server has no prompts."

        if command == "/prompt":
            if not parts:
                raise ValueError("Usage: /prompt <name> <arg1=value1> <arg2=value2>")
            name, *assignments = parts
            arguments = {}
            for assignment in assignments:
                key, _, value = assignment.partition("=")
                if not key or not value or key in arguments:
                    raise ValueError("Provide each prompt argument once as name=value.")
                arguments[key] = value

            result = await client.get_prompt(name, arguments=arguments)
            return "\n".join(
                message.content.text
                for message in result.messages if isinstance(message.content, TextContent)
            )

        # Keep direct tool discovery and calls available for comparison.
        if command == "/tools" and not parts:
            tools = await client.list_tools()
            lines = [f"{t.name}: {t.description}" for t in tools]
            return "\n".join(lines) or "The server has no tools."

        if command == "/call" and parts:
            tool_name, *assignments = parts
            arguments = {}
            for assignment in assignments:
                key, _, value = assignment.partition("=")
                if key in ("days", "max_results"):
                    arguments[key] = int(value)
                else:
                    arguments[key] = value
            call_res = await client.call_tool(tool_name, arguments)
            return str(call_res.data)

    raise ValueError(
        "Commands: @cities, @perth, @guidelines <category>, /prompts, "
        "/prompt <name> <arg=value>, /tools, /call <tool> <arg=value>"
    )


In [ ]:
async with Client(mcp) as client:
    # Preview the command results without sending them to a model.
    print(await handle_command(client, "@perth"))
    print(await handle_command(client, '/prompt weather_briefing city=Perth audience="photography students"'))


**Route input in the chat loop**

Use `query.startswith("@")` for resources and `query.startswith("/")` for slash commands. Inside the slash branch, check the command name. Send ordinary questions straight to `chat()`. A resource command prints its result. For `/prompt`, the client prints the rendered instructions, adds the relevant guide for the briefing or travel planner, and sends the resulting text to the model when LLM settings are configured. The existing chat method handles the model response and any tool calls.


In [ ]:
from fastmcp.exceptions import FastMCPError
from mcp import MCPError

PROMPT_GUIDES = {
    "weather_briefing": "comfort",
    "travel_weather_planner": "activity",
}


async def add_prompt_guide(client: Client, query: str, prompt: str) -> str:
    """Attach the reference guide requested by a rendered weather prompt."""
    name = shlex.split(query)[1]
    category = PROMPT_GUIDES.get(name)
    if category is None:
        return prompt
    guide = await handle_command(client, f"@guidelines {category}")
    return f"{prompt}\n\nReference guide:\n{guide}"


class WeatherResourceChatbot(MCPChatbot):
    """Add resource and prompt commands to the Section 5 chatbot."""

    async def chat_loop(self):
        """Route terminal commands before sending ordinary questions to the model."""
        print("\nMCP Chatbot Started!")
        print("Type your queries or 'quit' to exit.")
        print("Use @cities to list reference cities or @perth to read a city entry.")
        print("Use @guidelines <activity|comfort> to read a weather guide.")
        print("Use /prompts to list available prompts.")
        print("Use /prompt <name> <arg=value> to run a prompt.")
        print("Use /tools or /call <tool> <arg=value> to try tools directly.")

        while True:
            try:
                query = (await asyncio.to_thread(input, "\nQuery: ")).strip()
            except EOFError:
                break
            if not query:
                continue
            if query.lower() == "quit":
                break

            try:
                if query.startswith("@"):
                    print(await handle_command(self.client, query))
                elif query.startswith("/"):
                    command = query.split(maxsplit=1)[0].lower()
                    result = await handle_command(self.client, query)
                    print(result)
                    if command == "/prompt":
                        self.validate_settings()
                        await self.chat(await add_prompt_guide(self.client, query, result))
                else:
                    self.validate_settings()
                    await self.chat(query)
            except (ValueError, FastMCPError, MCPError, RuntimeError, requests.RequestException) as error:
                print(error)


### 7.6 Run the Terminal Chatbot

[Download the Session 5 weather MCP ZIP](practice/session-5-weather-mcp.zip){download="session-5-weather-mcp.zip"}. Extract it and open a terminal in the extracted folder. Keep the `practice/` and `sample_data/` folders together. The server reads `reference_cities.json` for the city resources and `activity.md` and `comfort.md` for the guides.

Activate your Python environment and install the dependencies.

```bat
python -m pip install -r practice/requirements.txt
```

Set `CHAT_API_KEY` in `practice/mcp_chatbot.py` for chat and prompt execution. Set `WEATHER_API_KEY` in `practice/weather_mcp_server_with_resources.py` to call the weather tools. You can read resources, list prompts and preview rendered instructions without API keys.

Run the terminal chatbot. It starts the server over stdio.

```bat
python practice/weather_chatbot_with_resources.py
```

| Command | Expected result |
|---|---|
| `@cities` | The full reference catalogue |
| `@perth` | Perth's climate and coordinates |
| `@guidelines comfort` | The temperature comfort guide |
| `/prompts` | Prompt names and their arguments |
| `/prompt` | Usage with the prompt name and argument format |
| `/prompt weather_briefing city=Perth` | A briefing for general travellers when API keys are configured |
| `/prompt weather_briefing city=Perth audience="photography students"` | A briefing tailored to photography students when API keys are configured |
| `/call get_weather city=Perth` | Current conditions for Perth from WeatherAPI (requires a key) |
| `What is the weather in Perth?` | A chat response using the weather tool when API keys are configured |

Type `quit` to exit the terminal chatbot.


### 7.7 Run the Web UI

The ZIP from Section 7.6 also includes `practice/web_chat.py`, `practice/chat_store.py` and `practice/index.html`. FastAPI serves the page and handles chat requests. The page sends each message as JSON and displays the reply when the turn finishes.

- `GET /` serves the chat page.
- `GET /chats` lists the browser's saved conversations. `GET /chats/{chat_id}` loads a selected conversation.
- `POST /chat` uses the same MCP command handler and weather chatbot as the terminal. It creates a conversation on the first successful message, or adds a turn to the selected conversation.

Python's built-in `sqlite3` module creates `weather_chat_history.sqlite3` in the extracted folder. The database keeps the visible messages and the model's conversation context. A browser cookie identifies which chats to show after a refresh or server restart.

The diagram shows how the browser connects to the chatbot host and how the host talks to SQLite, the LLM API and the weather MCP server. FastAPI serves the page at `GET /` before the browser starts requesting chat data.

![Connection diagram showing the browser linked to FastAPI over HTTP, the chatbot host linked to SQLite and the LLM API, and the MCP client linked over stdio to the weather server, which calls WeatherAPI over HTTPS.](images/web-chatbot-connection.png)

Read the message flow from top to bottom. The host takes the MCP command path, the model path, or both when a rendered prompt is sent to the model.

![Sequence diagram showing the browser loading history and sending a message to FastAPI, FastAPI handling MCP or model requests, then saving the turn to SQLite and returning JSON.](images/web-chat-message-flow.png)

1. The browser calls `GET /chats` to populate the sidebar. When you open a saved chat, `GET /chats/{chat_id}` loads its messages. FastAPI uses the browser ID from the cookie to find the matching records.
2. When you send a message, the browser calls `POST /chat`. FastAPI restores the model context for the selected chat or starts a fresh conversation for **New chat**.
3. The MCP client handles commands such as `@perth` and `/prompts` directly. An ordinary question or a rendered prompt sent for execution goes to the LLM API. If the model requests a weather tool, the client calls it over stdio. The MCP server queries WeatherAPI over HTTPS, then returns the tool result so the model can complete its answer.
4. FastAPI saves the visible messages and model context to SQLite, then returns the completed reply as JSON. The browser adds the reply to the conversation and updates the sidebar.

From the extracted folder, activate your Python environment and use the same dependency list as the terminal chatbot. If you already installed it in Section 7.6, go straight to the second command.

```bat
python -m pip install -r practice/requirements.txt
python practice/web_chat.py
```

`practice/` is the folder containing `web_chat.py`. Running the file directly lets Python find the other scripts in that folder. The web app also uses its own file location to find `index.html` and the weather server.

Open [http://127.0.0.1:8000](http://127.0.0.1:8000) and try `@perth`, `@guidelines comfort` or `/prompts`. Use **New chat** to start a separate conversation and **History** to reopen one. To ask a weather question or run a prompt through the model, configure the same API keys as in Section 7.6. Press **Ctrl+C** in the terminal to stop the web app.
